# A [RD](https://en.wikipedia.org/wiki/Rate%E2%80%93distortion_theory)-Comparison of "Stereo" Transforms

Note: some results provided by this notebook are only generated if the audio signal is stereo. This notebook should be run locally (do not run it in Google Colab, for example).

In [ ]:
import sounddevice as sd
import pywt # pip install pywavelets
import math
import numpy as np
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
from scipy import signal
import zlib
import pylab

In [ ]:
def plot(y, xlabel='', ylabel='', title=''):
    fig = plt.figure()
    ax = fig.add_subplot(111)
    ax.set_title(title)
    ax.grid()
    ax.xaxis.set_label_text(xlabel)
    ax.yaxis.set_label_text(ylabel)
    x = np.linspace(0, len(y)-1, len(y))
    ax.plot(x, y, '.', markersize=1)
    plt.show(block=False)

## Produce some sound ... for example, speak for 5 seconds
The audio signal will be used to compare the RD performance of the transforms.

In [ ]:
fs = 44100      # Sampling frequency
duration = 5.0  # seconds
x = sd.rec(int(duration * fs), samplerate=fs, channels=2, dtype=np.int16)
print("Say something!")
while sd.wait():
    pass
print("done")

In [ ]:
x.dtype

## This is your audio

In [ ]:
sd.play(x)
plot(x, "Sample", "Amplitude", "Audio Signal")

## Let's remove some samples from the begining

In [ ]:
x = x[50000:]

## This is your definitive audio sequence

In [ ]:
sd.play(x)
plot(x, "Sample", "Amplitude", "Audio Signal")

## Let's zoom the signal

In [ ]:
chunk = x[101000:102000]
plot(chunk, "Sample", "Amplitude", "Audio Signal (zoom)")

## Listen to the channel 0

In [ ]:
channel_0 = x[:, 0]
sd.play(channel_0)
plot(channel_0, "Sample", "Amplitude", "Channel 0")

## Listen to the channel 1

In [ ]:
channel_1 = x[:, 1]
sd.play(channel_1)
plot(channel_1, "Sample", "Amplitude", "Channel 1")

## Differences between the channels
Subband conformed by the $\mathbf{w}_1$ coefficients in the MST.

In [ ]:
np.all(channel_0 == channel_1)

In [ ]:
channel_0

In [ ]:
channel_1

In [ ]:
differences = channel_0.astype(np.int32) - channel_1

In [ ]:
differences.dtype

In [ ]:
sd.play((differences*10).astype(np.int16))
plot(differences, "Sample", "Amplitude", "Channel 0 - Channel 1")

## "Average"
Subband composed of the $\mathbf{w}_0$ coefficients in the MST.

In [ ]:
average = channel_0.astype(np.int32) + channel_1

In [ ]:
average.dtype

In [ ]:
sd.play(average.astype(np.int16))
plot(average, "Sample", "Amplitude", "Average")

## Is the "transform" reversible?
If so, we should be able to restore the original channels from the subbands.

In [ ]:
reconstructed_channel_0 = (average + differences)/2

In [ ]:
np.all(reconstructed_channel_0 == channel_0)

In [ ]:
reconstructed_channel_0 - channel_0

In [ ]:
reconstructed_channel_1 = (average - differences)/2

In [ ]:
np.all(reconstructed_channel_1 == channel_1)

## Some RD stuff

In [ ]:
def average_energy(x):
    #return np.sum(x[:, 0].astype(np.double)*x[:, 0].astype(np.double))/len(x[:, 0]) + \
    #       np.sum(x[:, 1].astype(np.double)*x[:, 1].astype(np.double))/len(x[:, 1])
    return np.sum(x.astype(np.double)*x.astype(np.double))/len(x)
    
def RMSE(x, y):
    error_signal = x - y
    return math.sqrt(average_energy(error_signal))

# Based on https://stackoverflow.com/questions/15450192/fastest-way-to-compute-entropy-in-python
def entropy_in_bits_per_symbol(sequence_of_symbols):
    value, counts = np.unique(sequence_of_symbols, return_counts = True)
    probs = counts / len(sequence_of_symbols)
    n_classes = np.count_nonzero(probs)

    if n_classes <= 1:
        return 0

    entropy = 0.
    for i in probs:
        entropy -= i * math.log(i, 2)

    return entropy

def deadzone_quantizer(x, quantization_step):
    k = (x / quantization_step).astype(np.int32)
    return k

def deadzone_dequantizer(k, quantization_step):
    y = quantization_step * k
    return y

def deadzone_qdeq(x, quantization_step):
    k = deadzone_quantizer(x, quantization_step)
    y = deadzone_dequantizer(k, quantization_step)
    return k, y

def RD_curve(data, analyze, synthesize):
    RD_points = []
    for q_step in range(16, 1024, 32):
        analyzed_data = analyze(data)
        k, y = deadzone_qdeq(analyzed_data, q_step)
        rate = entropy_in_bits_per_symbol(k[:, 0]) + entropy_in_bits_per_symbol(k[:, 1])
        reconstructed_data = synthesize(y)
        distortion = RMSE(data, reconstructed_data)
        RD_points.append((rate, distortion))
    return RD_points

## [Karhunen-Loéve Transform (KLT)](https://en.wikipedia.org/wiki/Karhunen%E2%80%93Lo%C3%A8ve_theorem)

The MST is not an optimal transform, from the decorrelation point of view.

<!-- ## [Karhunen-Loéve Transform (KLT)](https://en.wikipedia.org/wiki/Karhunen%E2%80%93Lo%C3%A8ve_theorem), [Hadamard](https://en.wikipedia.org/wiki/Hadamard_transform) and [Haar](https://en.wikipedia.org/wiki/Haar_wavelet#Haar_matrix) -->

The rows (i.e., the [basis](https://en.wikipedia.org/wiki/Basis_(linear_algebra)) vectors) of the of the [discrete](https://en.wikipedia.org/wiki/Discrete_transform) (all transforms discussed in this document are discrete) KLT consist of the [eigenvectors](https://en.wikipedia.org/wiki/Eigenvalues_and_eigenvectors#:~:text=In%20linear%20algebra%2C%20an%20eigenvector,which%20the%20eigenvector%20is%20scaled.) of the [autorrelation matrix](https://en.wikipedia.org/wiki/Autocorrelation#Auto-correlation_of_random_vectors) (also known as the [covariance](https://en.wikipedia.org/wiki/Covariance) matrix) of the transformed signal. In block-based [transform coding](https://web.stanford.edu/class/ee368b/Handouts/11-TransformCoding.pdf) (such as it happens in InterCom) it can be assumed that, on average, the amount of information provided by a subband (the set of coefficients that correspond to a given range of frequencies) is proportional to the variance in this subband, computed over the coefficients of the subband for different blocks.

The KLT minimizes the [geometric mean](https://en.wikipedia.org/wiki/Geometric_mean) of the variance of the transform coefficients. Hence, the KLT provides optimum energy concentration and therefore, the largest [Transform Coding Gain (TCG)](http://ws2.binghamton.edu/fowler/fowler%20personal%20page/EE523_files/Ch_13_3%20Transform%20Coding%20-%20Coding%20Gain%20&%20Classic%20Transforms%20(PPT).pdf) of any linear transform coding method. Unfortunately, the autocorrelation matrix depends on the input signal and therefore, the [change of basis](https://en.wikipedia.org/wiki/Change_of_basis) (the transform) provided by the KLT is signal dependent (each signal needs to be transformed by a different transform matrix), except in the case of 2-samples transform coding.

## TGC
The TCG is defined as the ratio of the arithmetic mean of the variances of the coefficients to their geometric mean:

\begin{equation}
G_{TC} = \frac{\frac{1}{N}\sum_{i=1}^N{\sigma_i^2}}{(\prod_{i=1}^N\sigma_i^2)^{\frac{1}{N}}}
\end{equation}

In the context of signal compression, variance, entropy and information are synonymous.

In [ ]:
def TCG(variances):
    N = len(variances)
    arithmetic_mean = np.sum(variances)/N
    geometric_mean = np.prod(variances)**(1/N)
    if geometric_mean == 0:
        geometric_mean = 1
    transform_coding_gain = arithmetic_mean/geometric_mean
    return transform_coding_gain

In [ ]:
# KLT analysis transform:
#
#  [w[0]] = 1/sqrt(2) [1  1] [x[0]]
#  [w[1]]             [1 -1] [x[1]]

def KLT_analyze(x):
    w = np.empty_like(x, dtype=np.int32)
    w[:, 0] = np.rint((x[:, 0].astype(np.int32) + x[:, 1]) / math.sqrt(2)) # L
    w[:, 1] = np.rint((x[:, 0].astype(np.int32) - x[:, 1]) / math.sqrt(2)) # H
    return w

# Synthesis transform:
#
#  [x[0]] = 1/sqrt(2) [1  1] [w[0]]
#  [x[1]]             [1 -1] [w[1]]

def KLT_synthesize(w):
    x = np.empty_like(w, dtype=np.int16)
    #x[:, 0] = np.rint((w[:, 0] + w[:, 1]) / math.sqrt(2)) # L(ow frequency subband)
    #x[:, 1] = np.rint((w[:, 0] - w[:, 1]) / math.sqrt(2)) # H(igh frequency subband)
    x[:, :] = KLT_analyze(w)
    return x

## Let's see the KLT subbands

In [ ]:
w = KLT_analyze(x)

## First subband (low frequencies)

In [ ]:
plot(w[:, 0], "Sample", "Amplitude", "Low-freq Subband ")
sd.play(w[:, 0].astype(np.int16))

## Second subband (high frequencies)

In [ ]:
plot(w[:, 1], "Sample", "Amplitude", "High-freq subband")
sd.play((w[:, 1]*10).astype(np.int16))

If you can listen to the low frequencies, this a consequence of the bad slope of the cut-off frequency in the high-frequency filter.

## Frequency vs time in the original signal

In [ ]:
plt.figure(figsize=(10, 6))

# Top Subplot - Left Channel
plt.subplot(2, 1, 1)
plt.title('Spectrogram — Left Channel')
plt.specgram(x[:, 0], Fs=44100)
plt.ylabel('Frequency (Hz)')

# Bottom Subplot - Right Channel
plt.subplot(2, 1, 2)
plt.title('Spectrogram — Right Channel')
plt.specgram(x[:, 1], Fs=44100)
plt.xlabel('Time (s)')
plt.ylabel('Frequency (Hz)')

plt.tight_layout()  # Prevents title overlaps between subplots
plt.show()

## Frequency vs time in the transformed signal

In [ ]:
plt.figure(figsize=(10, 6))

# Subplot 1: Low-Pass Filter
plt.subplot(2, 1, 1)
plt.title('KLT Low-Pass Filtered Signal (Low Frequencies)')
plt.specgram(w[:, 0], Fs=44100)
plt.ylabel('Frequency (Hz)')

# Subplot 2: High-Pass Filter
plt.subplot(2, 1, 2)
plt.title('KLT High-Pass Filtered Signal (High Frequencies)')
plt.specgram(w[:, 1], Fs=44100)
plt.xlabel('Time (s)')
plt.ylabel('Frequency (Hz)')

plt.tight_layout()  # Prevents text overlap between plots
plt.show()

You should see more energy of higher frequency (and some kind of attenuation in the low frequencies) in the high-frequency subband, and viceversa. 

## [Filter's response in the frequency domain](https://en.wikipedia.org/wiki/Filter_(signal_processing)#The_transfer_function)

In [ ]:
sqrt2 = math.sqrt(2)
K1 = np.array([sqrt2, sqrt2])
w1, h1 = signal.freqz(K1, fs=44100)
K2 = np.array([sqrt2, -sqrt2])
w2, h2 = signal.freqz(K2, fs=44100)

In [ ]:
plt.figure(figsize=(10, 6))

# Subplot 1: Low-Pass Filter Frequency Response
plt.subplot(2, 1, 1)
plt.title('KLT Low-Pass Filter Frequency Response')
plt.plot(w1, 20 * np.log10(abs(h1)), 'b')
plt.ylabel('Amplitude [dB]')
plt.grid(True)

# Subplot 2: High-Pass Filter Frequency Response
plt.subplot(2, 1, 2)
plt.title('KLT High-Pass Filter Frequency Response')
plt.plot(w2, 20 * np.log10(abs(h2)), 'b')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Amplitude [dB]')
plt.grid(True)

plt.tight_layout()  # Prevents title overlaps
plt.show()

As can be noticed, the [transition band](https://en.wikipedia.org/wiki/Transition_band) for these filters is very vage, in the sense that the transition between the pass band and the stop band is very wide (the transition band is very large). This implies that these filters are quite ineffective. However, this is the maximum performance that we can get using only two coeffients (or taps)/filter (remember that the KLT is an optimal transform, but obviosly, the spectral separation depends on the number of filters and their length).

Notice also that these filters are [Quadrature Mirror Filters (QMF)](https://en.wikipedia.org/wiki/Quadrature_mirror_filter). Thanks to this, even when there is a high degree of aliasing between the filters, the perfect reconstruction is possible.

## [Mid/Side Transform (MST)](https://en.wikipedia.org/wiki/Joint_encoding#M/S_stereo_coding)

MST represents the left and right channels as a *mid* subband (also sometimes called again channel) and a *side* subband. The mid subband is the sum of the left and right channels ($M=L+R$). The side subband is the difference of the left and right channels ($S=L-R$). See also Section 2.2.1. *Sum-Difference coding* at this [PhD thesis](https://research.tue.nl/en/studentTheses/stereo-coding-by-two-channel-linear-prediction-and-rotation).

In [ ]:
# Analysis transform:
#
#  [w[0]] = [1  1] [x[0]]
#  [w[1]]   [1 -1] [x[1]]

def MST_analyze(x):
    w = np.empty_like(x, dtype=np.int32)
    w[:, 0] = x[:, 0].astype(np.int32) + x[:, 1] # L(ow frequency subband)
    w[:, 1] = x[:, 0].astype(np.int32) - x[:, 1] # H(igh frequency subband)
    return w

# Inverse transform:
#
#  [x[0]] = 1/2 [1  1] [w[0]]
#  [x[1]]       [1 -1] [w[1]]

def MST_synthesize(w):
    x = np.empty_like(w, dtype=np.int16)
    x[:, 0] = (w[:, 0] + w[:, 1])/2 # L(ow frequency subband)
    x[:, 1] = (w[:, 0] - w[:, 1])/2 # H(igh frequency subband)
    return x

# This solution is slower than the previous one because: (1) an array of int32 is allocated,
# and (2) the memory locality is smaller (the division by 2 is not performed just when
# the samples are accesed).
def _MST_synthesize(w):
    x = np.empty_like(w, dtype=np.int16)
    x[:, :] = MST_analyze(w)
    x = x / 2
    return x

## MST subbands

In [ ]:
w = MST_analyze(x)

## Mid subband (low frequencies)

In [ ]:
plot(w[:, 0], "Sample", "Amplitude", "Mid Subband")
sd.play(w[:, 0].astype(np.int16))

## Side subband (high frequencies)

In [ ]:
plot(w[:, 1], "Sample", "Amplitude", "Side Subband")
sd.play((w[:, 1]*10).astype(np.int16))

## Filter's response in the frequency domain

In [ ]:
K1 = np.array([1.0, 1.0])
w1, h1 = signal.freqz(K1, fs=44100)
K2 = np.array([1.0, -1.0])
w2, h2 = signal.freqz(K2, fs=44100)

In [ ]:
plt.figure(figsize=(10, 6))

# Subplot 1: Low-Pass Filter Frequency Response
plt.subplot(2, 1, 1)
plt.title('MST Low-Pass Filter Frequency Response')
plt.plot(w1, 20 * np.log10(abs(h1)), 'b')
plt.ylabel('Amplitude [dB]')
plt.grid(True)

# Subplot 2: High-Pass Filter Frequency Response
plt.subplot(2, 1, 2)
plt.title('MST High-Pass Filter Frequency Response')
plt.plot(w2, 20 * np.log10(abs(h2)), 'b')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Amplitude [dB]')
plt.grid(True)

plt.tight_layout()  # Prevents title overlaps
plt.show()

The frequency response of the MST filters is identical to the freq. response of the 2x2 KLT.

## MST vs KLT from a RD perspective

In [ ]:
MST_RD_points = RD_curve(x, MST_analyze, MST_synthesize)

In [ ]:
KLT_RD_points = RD_curve(x, KLT_analyze, KLT_synthesize)

In [ ]:
plt.title("RD Tradeoff")
plt.xlabel("Bits per Sample")
plt.ylabel("RMSE")
#plt.xscale("log")
#plt.yscale("log")
plt.scatter(*zip(*MST_RD_points), c='b', marker="+", label='MST')
plt.scatter(*zip(*KLT_RD_points), c='r', marker="x", label='KLT')
plt.legend(loc='upper right')
plt.show()

The response is the same, obviously.

## Timmings

In [ ]:
%timeit MST_synthesize(MST_analyze(x))

In [ ]:
%timeit KLT_synthesize(KLT_analyze(x))

The MST should be faster.

## RD conclusions

MST and KLT are equivalent from a R/D perspective. However, MST is faster.

## Let's see the TGC

In [ ]:
LH = MST_analyze(x)
L = LH[:, 0]
H = LH[:, 1]
variances = [None]*2
variances[0] = np.var(L)
variances[1] = np.var(H)
print(TCG(variances))

In [ ]:
LH = KLT_analyze(x)
L = LH[:, 0]
H = LH[:, 1]
variances = [None]*2
variances[0] = np.var(L)
variances[1] = np.var(H)
print(TCG(variances))

The TCG of the KLT used to be sliiiightly better than the TCG of the MSC.

## S Transform (ST)
Another transform, only for the sake of curiosity.

In [ ]:
# https://www.researchgate.net/profile/Amir_Said2/publication/2598141_Reversible_Image_Compression_Via_Multiresolution_Representation_and_Predictive_Coding/links/56952cf008ae820ff074a536/Reversible-Image-Compression-Via-Multiresolution-Representation-and-Predictive-Coding.pdf
#
# Forward transform:
#
#  w[0] = ceil((x[0] + x[1])/2)
#  w[1] = x[0] - x[1] 
#
# Inverse transform:
#
#  x[0] = w[0] + ceil((w[1]+1)/2)
#  x[1] = x[0] - w[1]

def ST_analyze(x):
    w = np.empty_like(x, dtype=np.int32)
    w[:, 0] = np.ceil((x[:, 0].astype(np.int32) + x[:, 1])/2)
    w[:, 1] = x[:, 0].astype(np.int32) - x[:, 1]
    return w

def ST_synthesize(w):
    x = np.empty_like(w, dtype=np.int16)
    x[:, 0] = w[:, 0] + np.ceil((w[:, 1] + 1)/2)
    x[:, 1] = x[:, 0] - w[:, 1]
    return x

## ST vs MST

In [ ]:
ST_RD_points = RD_curve(x, ST_analyze, ST_synthesize)

In [ ]:
plt.title("R/D comparative")
plt.xlabel("Bits per sample")
plt.ylabel("RMSE")
#plt.xscale("log")
#plt.yscale("log")
plt.scatter(*zip(*MST_RD_points), c='b', marker="+", label='MST')
plt.scatter(*zip(*ST_RD_points), c='r', marker="x", label='ST')
plt.legend(loc='upper right')
plt.show()

MST performs slighly better than ST. This difference probably responds to numeric truncation issues during the transform computation.

## Let's see the RD curve using DEFLATE ...
... instead of the entropy.

In [ ]:
def chunk_stereo_DEFLATE_RD_curve(chunk, analyze, synthesize):
    '''RD curve of the chunk.'''
    RD_points = []
    for q_step in range(16, 1024, 32):
        analyzed_chunk = analyze(chunk)
        k, y = deadzone_qdeq(analyzed_chunk, q_step)
        subband_0 = k[:, 0].copy()
        subband_1 = k[:, 1].copy()
        rate = 8*(len(zlib.compress(subband_0)) + len(zlib.compress(subband_1))) / chunk.size
        #rate = len(zlib.compress(subband_0)) + len(zlib.compress(subband_1))
        #rate = entropy_in_bits_per_symbol(k[:, 0]) + entropy_in_bits_per_symbol(k[:, 1])
        reconstructed_chunk = synthesize(y)
        distortion = RMSE(chunk, reconstructed_chunk)
        RD_points.append((rate, distortion))
    return RD_points

In [ ]:
x.size

In [ ]:
MST_RD_points_DEFLATE = chunk_stereo_DEFLATE_RD_curve(x, MST_analyze, MST_synthesize)

In [ ]:
plt.title("RD comparative")
plt.xlabel("Bits Per Sample")
plt.ylabel("RMSE")
#plt.xscale("log")
#plt.yscale("log")
plt.scatter(*zip(*MST_RD_points), c='b', marker="+", label='MSC (entropy)')
plt.scatter(*zip(*MST_RD_points_DEFLATE), c='r', marker="+", label='MSC (DEFLATE)')
#plt.scatter(*zip(*KLT_RD_points), c='r', marker="x", label='KLT')
plt.legend(loc='upper right')
plt.show()

The RD curve using DEFLATE is similar to the one generated using the entropy, although better as we expected.

## Impact of  each subband in the distortion of the reconstructed chunk
Reconstructing the signal using only one subband (we compare the reconstructed signal and the original one).

In [ ]:
def subband_stereo_DEFLATE_RD_curve(chunk, analyze, synthesize):
    '''RD curves per subband.'''
    mid_RD_points = []
    side_RD_points = []
    for q_step in range(16, 1024, 32):
    #for q_step in range(1):
        #q_step = 16
        analyzed_chunk = analyze(chunk)
        k, y = deadzone_qdeq(analyzed_chunk, q_step)

        # We create side_RD_points by "deleting" subband 0, where mid lives.
        tmp = y.copy()
        tmp[:, 0] = 0
        #print(tmp)
        reconstructed_chunk = synthesize(tmp)
        distortion = RMSE(chunk, reconstructed_chunk)
        rate = len(zlib.compress(k[:, 1].copy())) / k[:, 1].size
        side_RD_points.append((rate, distortion))
        
        # Now create mid_RD_points by "deleting" subband 1, where side is.
        tmp = y.copy()
        tmp[:, 1] = 0
        #print(chunk)
        reconstructed_chunk = synthesize(tmp)
        distortion = RMSE(chunk, reconstructed_chunk)
        rate = len(zlib.compress(k[:, 0].copy())) / k[:, 0].size
        mid_RD_points.append((rate, distortion))
        
    return [mid_RD_points, side_RD_points]

In [ ]:
MST_RD_points = subband_stereo_DEFLATE_RD_curve(x, MST_analyze, MST_synthesize)

In [ ]:
plt.title("RD using only one subband")
plt.xlabel("Bits Per Sample")
plt.ylabel("RMSE")
#plt.xscale("log")
#plt.yscale("log")
plt.scatter(*zip(*MST_RD_points[0]), c='b', marker=".", label='Using only Mid')
plt.scatter(*zip(*MST_RD_points[1]), c='r', marker=".", label='Using only Side')
#plt.scatter(*zip(*KLT_RD_points), c='r', marker="x", label='KLT')
plt.legend(loc='upper right')
plt.show()
print("RMSE (using only zeros) =", RMSE(x, np.zeros(x.shape)))

### Another view of the same RD curves

In [ ]:
fig = plt.figure()

ax1 = fig.add_subplot(111)
ax1.scatter(*zip(*MST_RD_points[0]), c='b', marker=".", label='')
#ax1.legend(loc='upper right')
#ax1.set_title('$\Delta_0=\Delta_1$')
ax1.set_ylabel("RMSE (Only Mid Subband)", color='b')
ax1.set_xlabel("Bytes")
ax1.set_title("RMSE vs Bytes")

ax2 = ax1.twinx()
ax2.scatter(*zip(*MST_RD_points[1]), c='r', marker=".", label='')
#ax2.legend(loc='center right')
ax2.set_ylabel("RMSE (Only Side Subband)", color='r')

plt.show()

As it can be seen, the RD curve of the Mid subband has a larger impact on the quality of the reconstructed audio signal than than the Side channel (the dynamic range of the distortion is higher in the Mid subband). These results suggest that we should transmit more data of the Mid subband that of the Side subband. But what about the QSS used for each subband?

## So ... how we must quantize the subbans?

Ideally, in order to maximize the quality of the reconstructed signal, the quantization step for each subband should operate with identical RD slopes.

In [ ]:
#print(MST_RD_points[0])
slopes = [None]*2
for subband in range(2):
    slopes[subband] = []
    prev = (MST_RD_points[subband][::-1][0][0]+1, 0) # (Rate, Distortion)
    for i in MST_RD_points[subband][::-1]:
        #print(i, prev)
        try:
            slope = abs((i[1] - prev[1])/(i[0] - prev[0]))
            #print(i[1] - prev[1], i[0] - prev[0])
        except ZeroDivisionError:
            slope = 0.0
        #print(subband, slope)
        slopes[subband].append(slope)
        prev = i

pylab.scatter(np.linspace(0, len(slopes[0])-2, len(slopes[0])-1), slopes[0][1:], label="Mid", color='b')
pylab.plot(np.linspace(0, len(slopes[0])-2, len(slopes[0])-1), slopes[0][1:], color='b')
pylab.scatter(np.linspace(0, len(slopes[1])-2, len(slopes[1])-1), slopes[1][1:], label="Side", color='r')
pylab.plot(np.linspace(0, len(slopes[1])-2, len(slopes[1])-1), slopes[1][1:], color='r')
pylab.legend(loc='best')
pylab.title("RD slopes")
pylab.xlabel("Index of the RD point")
pylab.ylabel("Slope")
pylab.show()

Although this depends on the input audio signal, these results can suggests that we must send first all data of Mid and then, all data of Side (the most part of the information is in the Mid subband). Therefore, we should use $\Delta_\text{Mid} << \Delta_\text{Side}$.

## How good (or bad) is the quantization pattern $\Delta_\text{Mid} == \Delta_\text{Side}$ ... in this experiment?

In order to check the performance of this quantization heuristic (which assumes that both subbands have the same amount of information), let's compare two RD curves (in several different configurations). In the first one, $\Delta_0=\Delta_1$, for a set of QSSs, and in the second curve, we are going to use only one QSS for one of the subbands (for the other, the QSS has been fixed). If the heuristic were good, both curves should "touch" in one point, where also $\Delta_\text{Mid}=\Delta_\text{Side}$, and 

In [ ]:
def stereo_DEFLATE_RD_curve_same_deltas(chunk, analyze, synthesize):
    '''Both subbands are quantized using the same quantization step.'''
    RD_points = []
    for q_step in range(16, 2048, 32):
        analyzed_chunk = analyze(chunk)
        k, y = deadzone_qdeq(analyzed_chunk, q_step)
        k_0 = k[:, 0].copy()
        k_1 = k[:, 1].copy()
        rate = len(zlib.compress(k_0)) + len(zlib.compress(k_1))
        #rate = entropy_in_bits_per_symbol(k[:, 0]) + entropy_in_bits_per_symbol(k[:, 1])
        #print(y)
        reconstructed_chunk = synthesize(y)
        distortion = RMSE(chunk, reconstructed_chunk)
        RD_points.append((rate, distortion))
        #RD_points.append((q_step, distortion))
    return RD_points

def stereo_DEFLATE_RD_curve_fixing_delta0(chunk, analyze, synthesize, delta):
    '''Fixes delta_0 and varies delta_1.'''
    RD_points = []
    RQ_points = []
    for q_step in range(16, 2048, 32):
        analyzed_chunk = analyze(chunk)
        subband_0 = analyzed_chunk[:, 0].copy()
        subband_1 = analyzed_chunk[:, 1].copy()
        k_0, y_0 = deadzone_qdeq(subband_0, delta)
        k_1, y_1 = deadzone_qdeq(subband_1, q_step)
        rate = len(zlib.compress(k_0)) + len(zlib.compress(k_1))
        y = np.empty_like(x, dtype=np.int32)
        y[:, 0] = y_0
        y[:, 1] = y_1
        reconstructed_chunk = synthesize(y)
        distortion = RMSE(chunk, reconstructed_chunk)
        RD_points.append((rate, distortion))
        RQ_points.append((rate, q_step))
    return RD_points, RQ_points

def stereo_DEFLATE_RD_curve_fixing_delta1(chunk, analyze, synthesize, delta):
    '''Fixes delta_1 and varies delta_0.'''
    RD_points = []
    QD_points = []
    for q_step in range(16, 2048, 32):
        analyzed_chunk = analyze(chunk)
        subband_0 = analyzed_chunk[:, 0].copy()
        subband_1 = analyzed_chunk[:, 1].copy()
        k_0, y_0 = deadzone_qdeq(subband_0, q_step)
        k_1, y_1 = deadzone_qdeq(subband_1, delta)
        rate = len(zlib.compress(k_0)) + len(zlib.compress(k_1))
        y = np.empty_like(x, dtype=np.int32)
        y[:, 0] = y_0
        y[:, 1] = y_1
        reconstructed_chunk = synthesize(y)
        distortion = RMSE(chunk, reconstructed_chunk)
        RD_points.append((rate, distortion))
        QD_points.append((rate, q_step))
    return RD_points, QD_points

def stereo_DEFLATE_RD_point(chunk, analyze, synthesize, delta):
    '''Compute the RD point for delta_0=delta_1.'''
    RD_points = []
    analyzed_chunk = analyze(chunk)
    subband_0 = analyzed_chunk[:, 0].copy()
    subband_1 = analyzed_chunk[:, 1].copy()
    k_0, y_0 = deadzone_qdeq(subband_0, delta)
    k_1, y_1 = deadzone_qdeq(subband_1, delta)
    rate = len(zlib.compress(k_0)) + len(zlib.compress(k_1))
    y = np.empty_like(x, dtype=np.int32)
    y[:, 0] = y_0
    y[:, 1] = y_1
    reconstructed_chunk = synthesize(y)
    distortion = RMSE(chunk, reconstructed_chunk)
    RD_points.append((rate, distortion))
    return RD_points

In [ ]:
MST_RD_points_same_deltas = stereo_DEFLATE_RD_curve_same_deltas(x, MST_analyze, MST_synthesize)

## $\Delta=1024$

In [ ]:
delta = 1024
MST_RD_points_fixing_delta0, MST_RQ_points_fixing_delta0 = stereo_DEFLATE_RD_curve_fixing_delta0(x, MST_analyze, MST_synthesize, delta)
MST_RD_points_fixing_delta1, MST_RQ_points_fixing_delta1 = stereo_DEFLATE_RD_curve_fixing_delta1(x, MST_analyze, MST_synthesize, delta)

In [ ]:
MST_RD_point = stereo_DEFLATE_RD_point(x, MST_analyze, MST_synthesize, delta)

In [ ]:
fig = plt.figure()
ax1 = fig.add_subplot(111)
ax1.scatter(*zip(*MST_RD_point), c='b', marker="o", label=fr'$\Delta_0=\Delta_1={delta}$')
ax1.plot(*zip(*MST_RD_points_same_deltas), c='r', label=r'$\Delta_0=\Delta_1$')
ax1.scatter(*zip(*MST_RD_points_same_deltas), c='r', marker=".")
ax1.plot(*zip(*MST_RD_points_fixing_delta0), c='g', label=fr'$\Delta_0={delta}$, $\Delta_1$ varies')
ax1.scatter(*zip(*MST_RD_points_fixing_delta0), c='g', marker=".")
ax1.legend(loc='upper right')
ax1.set_ylabel("RMSE")
ax1.set_xlabel("Bytes")
ax1.set_title(r"Exploring Rates, Distortions and $\Delta$'s")

#ax2 = ax1.twinx()
#ax2.scatter(*zip(*MST_RQ_points_fixing_delta0), c='c', marker=".", label='$\Delta_1$')
#ax2.legend(loc='center right')
#ax2.set_ylabel("$\Delta_1$")
plt.show()

## $\Delta=512$

In [ ]:
delta = 512
MST_RD_points_fixing_delta0, MST_RQ_points_fixing_delta0 = stereo_DEFLATE_RD_curve_fixing_delta0(x, MST_analyze, MST_synthesize, delta)
MST_RD_points_fixing_delta1, MST_RQ_points_fixing_delta1 = stereo_DEFLATE_RD_curve_fixing_delta1(x, MST_analyze, MST_synthesize, delta)

In [ ]:
MST_RD_point = stereo_DEFLATE_RD_point(x, MST_analyze, MST_synthesize, delta)

In [ ]:
fig = plt.figure()
ax1 = fig.add_subplot(111)
ax1.scatter(*zip(*MST_RD_point), c='b', marker="o", label=fr'$\Delta_0=\Delta_1={delta}$')
ax1.plot(*zip(*MST_RD_points_same_deltas), c='r', label=r'$\Delta_0=\Delta_1$')
ax1.scatter(*zip(*MST_RD_points_same_deltas), c='r', marker='.')
ax1.plot(*zip(*MST_RD_points_fixing_delta0), c='g', label=fr'$\Delta_0={delta}$, $\Delta_1$ varies')
ax1.scatter(*zip(*MST_RD_points_fixing_delta0), c='g', marker='.')
ax1.legend(loc='upper right')
ax1.set_ylabel("RMSE")
ax1.set_xlabel("Bytes")
ax1.set_title(r"Exploring Rates, Distortions and $\Delta$'s")

#ax2 = ax1.twinx()
#ax2.scatter(*zip(*MST_RQ_points_fixing_delta0), c='c', marker=".", label='$\Delta_1$')
#ax2.legend(loc='center right')
#ax2.set_ylabel("$\Delta_1$")
plt.show()

## $\Delta=256$

In [ ]:
delta = 256
MST_RD_points_fixing_delta0, MST_RQ_points_fixing_delta0 = stereo_DEFLATE_RD_curve_fixing_delta0(x, MST_analyze, MST_synthesize, delta)
MST_RD_points_fixing_delta1, MST_RQ_points_fixing_delta1 = stereo_DEFLATE_RD_curve_fixing_delta1(x, MST_analyze, MST_synthesize, delta)

In [ ]:
MST_RD_point = stereo_DEFLATE_RD_point(x, MST_analyze, MST_synthesize, delta)

In [ ]:
fig = plt.figure()
ax1 = fig.add_subplot(111)
ax1.scatter(*zip(*MST_RD_point), c='b', marker="o", label=fr'$\Delta_0=\Delta_1={delta}$')
ax1.plot(*zip(*MST_RD_points_same_deltas), c='r', label=r'$\Delta_0=\Delta_1$')
ax1.scatter(*zip(*MST_RD_points_same_deltas), c='r', marker=".")
ax1.plot(*zip(*MST_RD_points_fixing_delta0), c='g', label=fr'$\Delta_0={delta}$, $\Delta_1$ varies')
ax1.scatter(*zip(*MST_RD_points_fixing_delta0), c='g', marker=".")
ax1.legend(loc='upper right')
ax1.set_ylabel("RMSE")
ax1.set_xlabel("Bytes")
ax1.set_title(r"Exploring Rates, Distortions and $\Delta$'s")

#ax2 = ax1.twinx()
#ax2.scatter(*zip(*MST_RQ_points_fixing_delta0), c='c', marker=".", label='$\Delta_1$')
#ax2.legend(loc='center right')
#ax2.set_ylabel("$\Delta_1$")
plt.show()

## $\Delta=128$

In [ ]:
delta = 128
MST_RD_points_fixing_delta0, MST_RQ_points_fixing_delta0 = stereo_DEFLATE_RD_curve_fixing_delta0(x, MST_analyze, MST_synthesize, delta)
MST_RD_points_fixing_delta1, MST_RQ_points_fixing_delta1 = stereo_DEFLATE_RD_curve_fixing_delta1(x, MST_analyze, MST_synthesize, delta)

In [ ]:
MST_RD_point = stereo_DEFLATE_RD_point(x, MST_analyze, MST_synthesize, delta)

In [ ]:
fig = plt.figure()
ax1 = fig.add_subplot(111)
ax1.scatter(*zip(*MST_RD_point), c='b', marker="o", label=fr'$\Delta_0=\Delta_1={delta}$')
ax1.plot(*zip(*MST_RD_points_same_deltas), c='r', label=r'$\Delta_0=\Delta_1$')
ax1.scatter(*zip(*MST_RD_points_same_deltas), c='r', marker='.')
ax1.plot(*zip(*MST_RD_points_fixing_delta0), c='g', label=fr'$\Delta_0={delta}$, $\Delta_1$ varies')
ax1.scatter(*zip(*MST_RD_points_fixing_delta0), c='g', marker='.')
ax1.legend(loc='upper right')
ax1.set_ylabel("RMSE")
ax1.set_xlabel("Bytes")
ax1.set_title(r"Exploring Rates, Distortions and $\Delta$'s")

#ax2 = ax1.twinx()
#ax2.scatter(*zip(*MST_RQ_points_fixing_delta0), c='c', marker=".", label='$\Delta_1$')
#ax2.legend(loc='center right')
#ax2.set_ylabel("$\Delta_1$")
plt.show()

## $\Delta=64$

In [ ]:
delta = 64
MST_RD_points_fixing_delta0, MST_RQ_points_fixing_delta0 = stereo_DEFLATE_RD_curve_fixing_delta0(x, MST_analyze, MST_synthesize, delta)
MST_RD_points_fixing_delta1, MST_RQ_points_fixing_delta1 = stereo_DEFLATE_RD_curve_fixing_delta1(x, MST_analyze, MST_synthesize, delta)

In [ ]:
MST_RD_point = stereo_DEFLATE_RD_point(x, MST_analyze, MST_synthesize, delta)

In [ ]:
fig = plt.figure()
ax1 = fig.add_subplot(111)
ax1.scatter(*zip(*MST_RD_point), c='b', marker="o", label=fr'$\Delta_0=\Delta_1={delta}$')
ax1.plot(*zip(*MST_RD_points_same_deltas), c='r', label=r'$\Delta_0=\Delta_1$')
ax1.scatter(*zip(*MST_RD_points_same_deltas), c='r', marker='.')
ax1.plot(*zip(*MST_RD_points_fixing_delta0), c='g', label=fr'$\Delta_0={delta}$, $\Delta_1$ varies')
ax1.plot(*zip(*MST_RD_points_fixing_delta0), c='g', marker='.')
ax1.legend(loc='upper right')
ax1.set_ylabel("RMSE")
ax1.set_xlabel("Bytes")
ax1.set_title(r"Exploring Rates, Distortions and $\Delta$'s")

#ax2 = ax1.twinx()
#ax2.scatter(*zip(*MST_RQ_points_fixing_delta0), c='c', marker=".", label='$\Delta_1$')
#ax2.legend(loc='center right')
#ax2.set_ylabel("$\Delta_1$")
plt.show()

Only for some intermediate values of the QSS, $\Delta_\text{Mid}\ne\Delta_\text{Side}$ works slighly better than $\Delta_\text{Mid}==\Delta_\text{Side}$.

The RD curves show that $\Delta_\text{Mid}==\Delta_\text{Side}$ works optimally (from a RD point of view, obviously). This is a consequence of that, although the forward transform is not energy preserving, the energy of the subbands is proportional to their contribution to the distortion, and therefore, under quantization, the MST behaves like an orthonormal one.

### RD slopes and filter's gain 

The subband gains in the MST suggest that $\Delta_\text{Mid}==\Delta_\text{Side}$ is not a bad option, because both subbands have the same gain ($\sqrt{2}$). Notice that, in general, the dynamic range of $w_0$ is much larger than $w_1$ and therefore, using this heuristic we will send the Side subband only after the most part of the energy of the Mid subband has been send. Remember that we are using a deadzone uniform (scalar) quantizer and therefore, "small" (in amplitude) signals tends to be completelly attenuated except for small $\Delta$'s.

<!-- For the sake of curiosity, it has been also plotted the distortion VS quantization step size, but this curve is not relevant for determining the quantization pattern between subbands. --> 

### Perceptual considerations

 At this point, we can found other perceptual reasons, such as to ensure the transmision of the stereo information, in which should be also hold that $\Delta_\text{Mid} == \Delta_\text{Side}$.